# 01 — Panel construction

Pulls raw CRSP data, cleans it (delisting-return merge), engineers features,
adds universe-membership flags, computes T+1 forward returns, and writes
the canonical `data/03_features/panel_daily.parquet`.

Run-once orchestrator. Subsequent notebooks read the panel; they never re-pull WRDS.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys; from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import wrds

from src import config, data, features


## 1. Pull universe permnos (top-2000 at 8 reference dates)


In [ ]:
WRDS_USER = 'YOUR_USERNAME'  # ← edit this
conn = wrds.Connection(wrds_username=WRDS_USER)

REFERENCE_DATES = ['1990-01-02', '1995-01-03', '2000-01-03', '2005-01-03',
                   '2010-01-04', '2015-01-05', '2020-01-02', '2023-01-03']
permnos = data.pull_universe_permnos(conn, REFERENCE_DATES, top_n=2000)
print(f'Total unique permnos: {len(permnos)}')


## 2. Pull raw DSF + DSEDELIST


In [ ]:
dsf = data.load_or_pull_dsf(conn, permnos, '1989-01-01', '2023-12-31')
dsedelist = data.load_or_pull_dsedelist(conn, permnos, '1989-01-01', '2023-12-31')
print(f'DSF: {len(dsf):,} rows, DSEDELIST: {len(dsedelist):,} rows')
conn.close()


## 3. Merge delisting returns into DSF (BUG-2 fix)


In [ ]:
daily_clean = data.merge_delisting_returns(dsf, dsedelist)
daily_clean.to_parquet(config.CLEAN_DIR / 'daily_clean.parquet', index=False)
print(f'daily_clean: {len(daily_clean):,} rows saved to {config.CLEAN_DIR}')


## 4. Feature engineering — the canonical panel


In [ ]:
panel = features.build_panel(daily_clean)
panel.to_parquet(config.PANEL_DAILY_PARQUET, index=False)
print(f'panel_daily.parquet: {len(panel):,} rows × {len(panel.columns)} cols')
print('Columns:', list(panel.columns)[:30], '...')


## 5. Quick sanity checks


In [ ]:
# Universe size over time
import matplotlib.pyplot as plt
panel.groupby('date')['in_top_2000'].sum().plot(
    figsize=(12, 3), title='Stocks in top-2000 over time'
)
plt.tight_layout()


In [ ]:
# Feature completeness in the OOS period (2010-2023)
oos = panel[panel['date'].dt.year.between(2010, 2023)]
for col in ['ret', 'vol_ewma_lag', 'mom_12_1', 'ewm_h60_skip', 'ret_fut_1m', 'in_top_2000']:
    valid_pct = oos[col].notna().mean() * 100
    print(f'  {col:<20} {valid_pct:5.1f}% non-null')
